In [1]:
import sqlite3
import pandas as pd
from cinesync.paths import DB_PATH

In [2]:
def counts_by(df, dims, normalize=False):
    if isinstance(dims, str):
        dims = [dims]
    out = df.groupby(dims, observed=True).size().rename("count").reset_index()
    if normalize:
        out["share"] = out["count"] / out["count"].sum()
    return out.sort_values("count", ascending=False)


def missing_by(df, has_col, dims):
    if isinstance(dims, str):
        dims = [dims]
    g = (
        df.groupby(dims, observed=True)[has_col]
        .agg(total="size", present="sum")
        .reset_index()
    )
    g["missing"] = g["total"] - g["present"]
    g["missing_rate"] = g["missing"] / g["total"]
    return g.sort_values("missing_rate", ascending=False)


def junction_long(con, table, value_col, df):
    """Read a junction table long and attach spine dimensions — for genre×language, keyword drills."""
    dims = df[
        ["title_id", "content_type", "original_language", "year_bucket", "decade"]
    ]
    long = pd.read_sql(f"SELECT title_id, {value_col} FROM {table}", con)
    return long.merge(dims, on="title_id", how="inner")

In [ ]:
_LB_BUCKETS = {
    "rating_0_5_count": 0.5,
    "rating_1_0_count": 1.0,
    "rating_1_5_count": 1.5,
    "rating_2_0_count": 2.0,
    "rating_2_5_count": 2.5,
    "rating_3_0_count": 3.0,
    "rating_3_5_count": 3.5,
    "rating_4_0_count": 4.0,
    "rating_4_5_count": 4.5,
    "rating_5_0_count": 5.0,
}


def load_spine(con):
    """One row per title: titles + exact child-table counts + credit/score presence + LB stats."""
    titles = pd.read_sql("SELECT * FROM titles", con)
    n_kw = pd.read_sql(
        "SELECT title_id, COUNT(*) AS n_keywords FROM title_keywords GROUP BY title_id",
        con,
    )
    n_gen = pd.read_sql(
        "SELECT title_id, COUNT(*) AS n_genres   FROM title_genres   GROUP BY title_id",
        con,
    )

    credits = pd.read_sql(
        "SELECT title_id, role, COUNT(*) AS n FROM title_credits "
        "WHERE role IN ('director','writer','creator') GROUP BY title_id, role",
        con,
    )
    credit_wide = (
        credits.pivot(index="title_id", columns="role", values="n")
        .add_prefix("n_")
        .reset_index()
        if len(credits)
        else pd.DataFrame({"title_id": []})
    )

    scores = pd.read_sql("SELECT title_id, source, score FROM title_scores", con)
    score_wide = (
        scores.pivot_table(
            index="title_id", columns="source", values="score", aggfunc="max"
        ).reset_index()
        if len(scores)
        else pd.DataFrame({"title_id": []})
    )

    bucket_cols = ", ".join(_LB_BUCKETS)
    lb = pd.read_sql(
        f"SELECT title_id, rating_value AS letterboxd_rating, rating_count AS lb_rating_count, "
        f"review_count AS lb_review_count, top_rank AS lb_top_rank, watches AS lb_watches, "
        f"likes AS lb_likes, lists AS lb_lists, {bucket_cols} FROM title_letterboxd_stats",
        con,
    )

    df = (
        titles.merge(n_kw, on="title_id", how="left")
        .merge(n_gen, on="title_id", how="left")
        .merge(credit_wide, on="title_id", how="left")
        .merge(score_wide, on="title_id", how="left")
        .merge(lb, on="title_id", how="left")
    )

    # lb_implied_rating: naive weighted-histogram mean (0-5), fallback for the withheld official rating
    b = df[list(_LB_BUCKETS)].fillna(0)
    denom = b.sum(axis=1)
    df["lb_implied_rating"] = (
        (sum(b[c] * w for c, w in _LB_BUCKETS.items()) / denom)
        .where(denom > 0)
        .round(3)
    )
    df = df.drop(columns=list(_LB_BUCKETS))

    for col in ["n_keywords", "n_genres", "n_director", "n_writer", "n_creator"]:
        if col in df:
            df[col] = df[col].fillna(0).astype(int)
    return df

In [ ]:
def add_derived(df):
    df = df.copy()
    yb = (df["release_year"] // 5) * 5
    df["year_bucket"] = pd.Categorical(
        yb.astype(int).astype(str) + "-" + (yb + 4).astype(int).astype(str),
        ordered=True,
        categories=[f"{s}-{s + 4}" for s in range(1940, 2026, 5)],
    )
    df["decade"] = ((df["release_year"] // 10) * 10).astype(int).astype(str) + "s"

    def txt(col):  # text present: not-null and non-empty (absent col -> all False)
        return (
            (df[col].notna() & (df[col].astype("string").str.len() > 0))
            if col in df
            else pd.Series(False, index=df.index)
        )

    def cnt(col):  # count column > 0 (absent col -> all False)
        return (
            (df[col].fillna(0) > 0) if col in df else pd.Series(False, index=df.index)
        )

    df["has_imdb_id"] = txt("imdb_id")
    df["has_wikidata"] = txt("wikidata_id")
    df["has_certificate"] = txt("certificate")
    df["has_tmdb_overview"] = txt("tmdb_overview")  # primary synopsis
    df["has_detailed_plot"] = txt("detailed_plot")  # enrichment scaffolding
    df["has_keywords"] = cnt("n_keywords")
    df["has_director"] = cnt("n_director")
    df["has_writer"] = cnt("n_writer")
    df["has_creator"] = cnt("n_creator")
    df["movie_missing_key_credit"] = df["content_type"].eq("movie") & (
        ~df["has_director"] | ~df["has_writer"]
    )
    for s in ["tmdb_rating", "rt_critic", "rt_audience", "imdb_rating"]:
        df[f"has_{s}"] = txt(s)
    df["has_letterboxd_rating"] = txt("letterboxd_rating")
    df["has_lb_implied"] = (
        df["lb_implied_rating"].notna()
        if "lb_implied_rating" in df
        else pd.Series(False, index=df.index)
    )
    df["has_lb_any"] = df["has_letterboxd_rating"] | df["has_lb_implied"]
    df["in_lb_top500"] = df["lb_top_rank"].notna() if "lb_top_rank" in df else False
    return df


In [5]:
conn = sqlite3.connect(DB_PATH)
spine = load_spine(conn)